In [0]:
import re
import pyspark.sql.functions as F
from pyspark.sql.functions import col, get_json_object, regexp_replace
from delta.tables import DeltaTable

# Data Cleansing

In [0]:
df = spark.read.table("fraud_detection_project.bronze_layer.customer_profiles")

# Standardize column names
def StandardizeNames(df):
    l = df.columns                                                  # (Regex Operator -> https://regex101.com/)
    cols = [re.sub(r'(?<!^)(?=[A-Z])', '_', c).lower() for c in l]  # Convert CamelCase to snake_case
    cols = [c.lstrip('_') for c in cols]  # Remove underscores in the beginning of column names
    return df.toDF(*cols)
df = StandardizeNames(df)

In [0]:
# Deleting duplicated data
df.dropDuplicates(['customer_id'])

# Deleting rows without some features
df = df.dropna(how='any', subset=['customer_id','file_path','ingest_datetime'])

In [0]:
# {"card_brand": "Mastercard", "card_category": "Black", "card_type": "D\u00e9bito", "security_code": "969", "issue_date": "2025-02-20", "expiration_date": "2028-09-30", "card_limit": 40172.24, "available_limit": 11102.13}

# Extracting directly the keys from card_details in the JSON
df = df.withColumn("card_brand",      get_json_object(col("card_details"), "$.card_brand"))
df = df.withColumn("card_category",   get_json_object(col("card_details"), "$.card_category"))
df = df.withColumn("card_type",       get_json_object(col("card_details"), "$.card_type"))
df = df.withColumn("security_code",   get_json_object(col("card_details"), "$.security_code"))
df = df.withColumn("issue_date",      get_json_object(col("card_details"), "$.issue_date"))
df = df.withColumn("expiration_date", get_json_object(col("card_details"), "$.expiration_date"))
df = df.withColumn("card_limit",      get_json_object(col("card_details"), "$.card_limit"))
df = df.withColumn("available_limit", get_json_object(col("card_details"), "$.available_limit"))
if 'is_card_virtual' in df.columns:
    df = df.withColumn("is_card_virtual", get_json_object(col("card_details"), "$.is_card_virtual"))

df.dtypes

In [0]:
# {"customer_gender": "M", "customer_age": 84}

# Extracting directly the keys from client_details in the JSON
df = df.withColumn("customer_gender", get_json_object(col("client_details"), "$.customer_gender"))
df = df.withColumn("customer_age",    get_json_object(col("client_details"), "$.customer_age"))
df.dtypes

In [0]:
# Deleting the column card_details
df = df.drop("card_details", "client_details")

In [0]:
# Deleting "-" caracters in the zip code
df = df.withColumn("customer_zip_code", F.regexp_replace(F.col("customer_zip_code"), "-", ""))

In [0]:
%skip
df.createOrReplaceTempView("df1")

In [0]:
%sql
CREATE OR REPLACE FUNCTION mask_card_number_fixed(card_number STRING)
RETURN concat('****-****-****-', right(card_number, 4));

CREATE OR REPLACE FUNCTION mask_cvv_fixed(cvv STRING)
RETURN '***';

In [0]:
%skip
ALTER TABLE fraud_detection_project.silver_layer.customer_profiles
ALTER COLUMN card_number SET MASK mask_card_number_fixed;

ALTER TABLE fraud_detection_project.silver_layer.customer_profiles
ALTER COLUMN security_code SET MASK mask_cvv_fixed;

In [0]:
target = "fraud_detection_project.silver_layer.customer_profiles"

if spark.catalog.tableExists(target):
    dt = DeltaTable.forName(spark, target)

    dt.alias("t").merge(
        df.alias("s"),
        "t.customer_id = s.customer_id"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()
    print("Merge concluded.")
else:
    df.write.format("delta") \
      .option("mergeSchema", "true") \
      .saveAsTable(target)
    print("Tabel created.")

In [0]:
%sql
SELECT count(*) FROM fraud_detection_project.silver_layer.customer_profiles